## Store vector to Postgres

#### *Issues encountered*
- Model(llama3) not support embedding, need use embedding model
- VECTOR_SIZE not match model dimensions, different model output different dimensions, need to match table's dimensions

In [ ]:
%pip install langchain langchain-docling

In [ ]:
%pip install langchain-postgres

In [ ]:
%pip install langchain-core langchain-ollama sqlalchemy

### Prepare Ollama

Since this example use Ollama embedding, we need prepare Ollama service ahead.

```bash
brew install ollama
brew services start ollama
ollama pull llama3
```

### Load env

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
print(os.getenv('POSTGRES_PORT'))
print(os.getenv('POSTGRES_USER'))
print(os.getenv('TEST'))

### Postgres connection string

In [ ]:
import os

print(os.getenv('POSTGRES_PORT'))
CONNECTION_STRING = (
    f"postgresql+asyncpg://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}"
    f":{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)

### Create Engine for store, and init table

In [ ]:
from langchain_postgres import PGEngine

pg_engine = PGEngine.from_connection_string(url=CONNECTION_STRING)





### Run only once

In [ ]:
await pg_engine.ainit_vectorstore_table(
    table_name=os.getenv('TABLE_NAME'),
    vector_size=os.getenv('VECTOR_SIZE'),
)

### Create Embedding for store

In [ ]:
from langchain_ollama import OllamaEmbeddings

# use llama3 to embedding cause error
# embedding = OllamaEmbeddings(model="llama3")

# use embed model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text", # dimension=768
    #base_url="http://127.0.0.1:11434",
    #dimensions=1024
)

In [ ]:
import ollama
ollama.embeddings(model='nomic-embed-text', prompt='The sky is blue because of rayleigh scattering')

### Create store

In [ ]:
from langchain_postgres import PGVectorStore

store = await PGVectorStore.create(
    engine=pg_engine,
    table_name=os.getenv('TABLE_NAME'),
    # schema_name=SCHEMA_NAME,
    embedding_service=embeddings,
    id_column="langchain_id"
)

In [ ]:
import uuid

from langchain_core.documents import Document

docs = [
    Document(
        id=str(uuid.uuid4()),
        page_content="Red Apple",
        metadata={"description": "red", "content": "1", "category": "fruit"},
    ),
    Document(
        id=str(uuid.uuid4()),
        page_content="Banana Cavendish",
        metadata={"description": "yellow", "content": "2", "category": "fruit"},
    ),
    Document(
        id=str(uuid.uuid4()),
        page_content="Orange Navel",
        metadata={"description": "orange", "content": "3", "category": "fruit"},
    ),
]

await store.aadd_documents(docs)

In [ ]:
import uuid

all_texts = ["Apples and oranges", "Cars and airplanes", "Pineapple", "Train", "Banana"]
metadatas = [{"len": len(t)} for t in all_texts]
ids = [str(uuid.uuid4()) for _ in all_texts]

await store.aadd_texts(all_texts, metadatas=metadatas, ids=ids)
#await store.aadd_texts(all_texts, metadatas=metadatas)